In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import entropy, kurtosis, skew

# Load the color image (in BGR format)
image = cv2.imread('filepath')

# Crop the image to 70% of its original size, centered and maintaining the same aspect ratio
height, width = image.shape[:2]
crop_ratio = 0.7
new_width = int(width * crop_ratio)
new_height = int(height * crop_ratio)
x_start = int((width - new_width) / 2)
y_start = int((height - new_height) / 2)
# Crop the image
cropped_image = image[y_start:y_start + new_height, x_start:x_start + new_width]

# Convert the cropped image to grayscale for further analysis
gray_image = cv2.cvtColor(cropped_image, cv2.COLOR_BGR2GRAY)

# Option to apply a simple contrast adjustment (no adjustment at alpha=1 and beta=0)
alpha = 1  # Contrast control (1.0-3.0)
beta = 0   # Brightness control (0-100)
adjusted_image = cv2.convertScaleAbs(gray_image, alpha=alpha, beta=beta)

# Display the cropped grayscale image
plt.figure(figsize=(6, 6))
plt.imshow(adjusted_image, cmap='gray')
plt.title('Cropped Grayscale Image Used for Analysis')
plt.axis('off')  # Hide axis for better visualization
plt.show()

# Blob detection setup
# Set up SimpleBlobDetector parameters.
params = cv2.SimpleBlobDetector_Params()

# Adjust thresholds
params.minThreshold = 0
params.maxThreshold = 255
params.thresholdStep = 5  # Smaller step for finer detection

# Filter by area
params.filterByArea = True
params.minArea = 500      # Increased to ignore smaller blobs
params.maxArea = 1e9      # Increased to include larger blobs

# Enable filtering by circularity
params.filterByCircularity = True
params.minCircularity = 0.1  # Adjusted to include blobs that are not perfect circles

# Enable filtering by convexity
params.filterByConvexity = True
params.minConvexity = 0.5  # Adjusted to include less convex blobs

# Enable filtering by inertia (elongation)
params.filterByInertia = True
params.minInertiaRatio = 0.1  # Adjusted to include elongated blobs

# Disable filtering by blob color
params.filterByColor = False

# Create a detector with the parameters
detector = cv2.SimpleBlobDetector_create(params)

# Detect blobs in the adjusted cropped image
keypoints = detector.detect(adjusted_image)

# Create a copy of the cropped color image to draw on (in BGR format)
im_with_keypoints = cropped_image.copy()

# Create a list to store mean grey values of blobs
mean_grey_values = []

# Draw detected blobs as circles and calculate mean grey values
for keypoint in keypoints:
    x, y = keypoint.pt  # Blob centroid
    diameter = keypoint.size  # Size of the blob
    radius = int(diameter / 2)
    # Draw circle around each blob in blue color (BGR)
    cv2.circle(im_with_keypoints, (int(x), int(y)), radius, (255, 0, 0), 2)  # Blue circles
    
    # Create a mask for the blob
    mask = np.zeros_like(adjusted_image, dtype=np.uint8)
    cv2.circle(mask, (int(x), int(y)), radius, 255, thickness=-1)  # White filled circle
    
    # Calculate mean grey value within the blob
    mean_val = cv2.mean(adjusted_image, mask=mask)[0]
    mean_grey_values.append(mean_val)

# Convert the image to RGB for Matplotlib visualization
im_with_keypoints_rgb = cv2.cvtColor(im_with_keypoints, cv2.COLOR_BGR2RGB)

# Show the image with manually drawn blobs
plt.figure(figsize=(8, 8))
plt.imshow(im_with_keypoints_rgb)
plt.title('Blobs Detected in the Cropped Image')
plt.axis('off')
plt.show()

# Number of blobs detected
num_blobs = len(keypoints)
print(f"Number of blobs detected: {num_blobs}")

# Collect diameters and areas of the detected blobs
diameters = []
areas = []

for keypoint in keypoints:
    diameter = keypoint.size  # Size of the blob
    area = np.pi * (diameter / 2) ** 2  # Area of the blob
    diameters.append(diameter)
    areas.append(area)

# Convert lists to numpy arrays for statistical calculations
diameters = np.array(diameters)
areas = np.array(areas)
mean_grey_values = np.array(mean_grey_values)

# Calculate summary statistics if blobs are detected
if num_blobs > 0:
    # Statistics for diameters
    avg_diameter = np.mean(diameters)
    std_diameter = np.std(diameters)
    min_diameter = np.min(diameters)
    max_diameter = np.max(diameters)
    median_diameter = np.median(diameters)

    # Statistics for areas
    avg_area = np.mean(areas)
    std_area = np.std(areas)
    min_area = np.min(areas)
    max_area = np.max(areas)
    median_area = np.median(areas)
    
    # Statistics for mean grey values
    avg_mean_grey = np.mean(mean_grey_values)
    std_mean_grey = np.std(mean_grey_values)
    min_mean_grey = np.min(mean_grey_values)
    max_mean_grey = np.max(mean_grey_values)
    median_mean_grey = np.median(mean_grey_values)

    print("\nDiameter Statistics (pixels):")
    print(f" - Average Diameter: {avg_diameter:.2f}")
    print(f" - Standard Deviation: {std_diameter:.2f}")
    print(f" - Minimum Diameter: {min_diameter:.2f}")
    print(f" - Maximum Diameter: {max_diameter:.2f}")
    print(f" - Median Diameter: {median_diameter:.2f}")

    print("\nArea Statistics (square pixels):")
    print(f" - Average Area: {avg_area:.2f}")
    print(f" - Standard Deviation: {std_area:.2f}")
    print(f" - Minimum Area: {min_area:.2f}")
    print(f" - Maximum Area: {max_area:.2f}")
    print(f" - Median Area: {median_area:.2f}")
    
    print("\nMean Grey Value Statistics within Blobs:")
    print(f" - Average Mean Grey Value: {avg_mean_grey:.2f}")
    print(f" - Standard Deviation: {std_mean_grey:.2f}")
    print(f" - Minimum Mean Grey Value: {min_mean_grey:.2f}")
    print(f" - Maximum Mean Grey Value: {max_mean_grey:.2f}")
    print(f" - Median Mean Grey Value: {median_mean_grey:.2f}")
else:
    print("No blobs detected to compute statistics.")

# Texture analysis on the cropped image
# Calculate the mean grey value (visual density) of the entire cropped image
mean_grey_value = np.mean(adjusted_image)
print(f"\nMean grey value (visual density) of the cropped image: {mean_grey_value}")

# Calculate texture features using histogram and simple statistics
hist = cv2.calcHist([adjusted_image], [0], None, [256], [0, 256])
hist = hist.flatten()

# Normalize the histogram to get probabilities
hist_norm = hist / hist.sum()

# Contrast: Standard deviation of the pixel values
contrast = np.std(adjusted_image)
print(f"Contrast: {contrast}")

# Energy: Sum of squared elements in the normalized histogram
energy = np.sum(hist_norm ** 2)
print(f"Energy: {energy}")

# Homogeneity: Calculated using the normalized histogram
homogeneity = np.sum(hist_norm / (1 + np.arange(256)))
print(f"Homogeneity: {homogeneity}")

# Entropy: Measure of randomness in the image
entropy_value = -np.sum(hist_norm * np.log2(hist_norm + 1e-10))
print(f"Entropy: {entropy_value}")

# Variance: Variance of the pixel values
variance = np.var(adjusted_image)
print(f"Variance: {variance}")

# Skewness: Measure of asymmetry of the pixel intensity distribution
skewness = skew(adjusted_image.flatten())
print(f"Skewness: {skewness}")

# Kurtosis: Measure of the "tailedness" of the pixel intensity distribution
kurt = kurtosis(adjusted_image.flatten())
print(f"Kurtosis: {kurt}")

# Sum of Squares (Variance of the histogram)
mean_hist_value = np.mean(hist)
sum_of_squares = np.sum((hist - mean_hist_value) ** 2)
print(f"Sum of Squares (Variance of histogram): {sum_of_squares}")

# Sum Average: Average intensity based on the histogram
sum_average = np.sum(np.arange(256) * hist_norm)
print(f"Sum Average: {sum_average}")

# Sum Entropy: Entropy of the sum distribution (same as image entropy here)
sum_entropy = -np.sum(hist_norm * np.log2(hist_norm + 1e-10))
print(f"Sum Entropy: {sum_entropy}")

# Difference Entropy: Entropy of the difference distribution
diff_hist = np.abs(hist[:-1] - hist[1:])
diff_hist_norm = diff_hist / diff_hist.sum()
difference_entropy = -np.sum(diff_hist_norm * np.log2(diff_hist_norm + 1e-10))
print(f"Difference Entropy: {difference_entropy}")

# Autocorrelation: Measure of repeating patterns in the histogram
autocorr = np.correlate(hist_norm, hist_norm, mode='full')
autocorr_value = autocorr[autocorr.size // 2]
print(f"Autocorrelation: {autocorr_value}")